In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")

db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

print("Koneksi ke database Kopi Nusantara berhasil!")


Koneksi ke database Kopi Nusantara berhasil!


A. Membangun Katalog (Create) (bobot 20%)

In [4]:
data_menu = [
    {"nama_menu": "Espresso", "harga": 15000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 20},
    {"nama_menu": "Americano", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 25},
    {"nama_menu": "Cappuccino", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 15},
    {"nama_menu": "Cafe Latte", "harga": 23000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 20},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 30},
    {"nama_menu": "Mocha", "harga": 27000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 12},

    {"nama_menu": "Roti Bakar", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 20},
    {"nama_menu": "Kentang Goreng", "harga": 18000, "kategori": "Makanan", "pedas_level": 1, "stok": 25},
    {"nama_menu": "Mie Goreng", "harga": 20000, "kategori": "Makanan", "pedas_level": 3, "stok": 15},
    {"nama_menu": "Nasi Goreng", "harga": 22000, "kategori": "Makanan", "pedas_level": 2, "stok": 18},
    {"nama_menu": "Pisang Goreng", "harga": 12000, "kategori": "Makanan", "pedas_level": 0, "stok": 30},
    {"nama_menu": "Cireng", "harga": 14000, "kategori": "Makanan", "pedas_level": 2, "stok": 25}
]

print("Jumlah dokumen di collection 'menu':", koleksi_menu.count_documents({}))

Jumlah dokumen di collection 'menu': 12


B. Kueri Bertingkat (Read) (bobot 20%)

In [12]:
# B1. Menampilkan menu Kopi dengan tingkat kepahitan di atas 3
print("Menu Kopi dengan tingkat kepahitan di atas 3:")
for data in db.menu.find({
    "kategori": "Kopi",
    "tingkat_kepahitan": {"$gt": 3}
}, {"_id": 0}):
    print(data)


# B2. Menampilkan menu dengan harga 15000 sampai 30000
print("\nMenu dengan harga 15000 sampai 30000:")
for data in db.menu.find({
    "harga": {"$gte": 15000, "$lte": 30000}
}, {"_id": 0}):
    print(data)


# B3. Menampilkan menu yang tidak memiliki field pedas_level
print("\nMenu yang tidak memiliki pedas_level:")
for data in db.menu.find({
    "pedas_level": {"$exists": False}
}, {"_id": 0}):
    print(data)


Menu Kopi dengan tingkat kepahitan di atas 3:
{'nama_menu': 'Espresso', 'harga': 15000, 'kategori': 'Kopi', 'tingkat_kepahitan': 5, 'stok': 20, 'status': 'Tersedia'}
{'nama_menu': 'Americano', 'harga': 18000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 25, 'status': 'Tersedia'}

Menu dengan harga 15000 sampai 30000:
{'nama_menu': 'Espresso', 'harga': 15000, 'kategori': 'Kopi', 'tingkat_kepahitan': 5, 'stok': 20, 'status': 'Tersedia'}
{'nama_menu': 'Americano', 'harga': 18000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 25, 'status': 'Tersedia'}
{'nama_menu': 'Cappuccino', 'harga': 25000, 'kategori': 'Kopi', 'tingkat_kepahitan': 3, 'stok': 15, 'status': 'Tersedia'}
{'nama_menu': 'Cafe Latte', 'harga': 23000, 'kategori': 'Kopi', 'tingkat_kepahitan': 2, 'stok': 20, 'status': 'Tersedia'}
{'nama_menu': 'Kopi Susu Gula Aren', 'harga': 22000, 'kategori': 'Kopi', 'tingkat_kepahitan': 2, 'stok': 30, 'status': 'Tersedia'}
{'nama_menu': 'Mocha', 'harga': 27000, 'kategori': 'Kopi',

C. Pembaruan Data (Update) (bobot 15%)

In [17]:
# C1. Menambahkan status Tersedia ke semua menu
hasil = db.menu.update_many(
    {},
    {"$set": {"status": "Tersedia"}}
)

print("Jumlah menu yang diperbarui:", hasil.modified_count)


# C2. Mengubah status Americano menjadi Habis
hasil1 = db.menu.update_one(
    {"nama_menu": "Americano"},
    {"$set": {"status": "Habis"}}
)

# Mengubah status Cappuccino menjadi Habis
hasil2 = db.menu.update_one(
    {"nama_menu": "Cappuccino"},
    {"$set": {"status": "Habis"}}
)

print("Jumlah menu yang diubah menjadi Habis:",
      hasil1.modified_count + hasil2.modified_count)


Jumlah menu yang diperbarui: 0
Jumlah menu yang diubah menjadi Habis: 2


D. Integrasi ke Spark (bobot 25%)

In [18]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

koleksi_menu = db.menu

# Langkah 1: Mengambil seluruh dokumen dari MongoDB
dokumen_menu = list(koleksi_menu.find({}))

# Langkah 2: Mengubah data menjadi Pandas DataFrame
df_pandas = pd.DataFrame(dokumen_menu)

# Langkah 3: Mengubah _id menjadi string
df_pandas["_id"] = df_pandas["_id"].astype(str)

# Menangani field yang tidak dimiliki semua dokumen
for kolom in ["tingkat_kepahitan", "pedas_level", "status"]:
    if kolom not in df_pandas.columns:
        df_pandas[kolom] = None
    else:
        df_pandas[kolom] = df_pandas[kolom].astype(object)
        df_pandas.loc[pd.isna(df_pandas[kolom]), kolom] = None

# Langkah 4: Mengubah Pandas DataFrame menjadi Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

# Menampilkan struktur dan isi Spark DataFrame
print("Struktur DataFrame:")
df_spark_menu.printSchema()

print("Isi DataFrame:")
df_spark_menu.show()

# Langkah 5: Menghitung rata-rata harga berdasarkan kategori
print("Rata-rata harga per kategori:")
df_spark_menu.groupBy("kategori").avg("harga").show()


Struktur DataFrame:
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: long (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)

Isi DataFrame:
+--------------------+-------------------+-----+--------+-----------------+----+--------+-----------+
|                 _id|          nama_menu|harga|kategori|tingkat_kepahitan|stok|  status|pedas_level|
+--------------------+-------------------+-----+--------+-----------------+----+--------+-----------+
|6aeb83048b26acc03...|           Espresso|15000|    Kopi|              5.0|  20|Tersedia|       NULL|
|6aeb83048b26acc03...|          Americano|18000|    Kopi|              4.0|  25|   Habis|       NULL|
|6aeb83048b26acc03...|         Cappuccino|25000|    Kopi|              3.0|  15|   Habis|       NULL|
|6aeb83048b26acc03...|         C

[Stage 7:============================================>              (3 + 1) / 4]

+--------+------------------+
|kategori|        avg(harga)|
+--------+------------------+
|    Kopi|21666.666666666668|
| Makanan|16833.333333333332|
+--------+------------------+



E. Kesimpulan (bobot 20%)

F. EXPLORASI

1. Menghitung Jumlah Menu Berdasarkan Kategori

In [19]:
# Menghitung jumlah menu setiap kategori
hasil = db.menu.aggregate([
    {"$group": {"_id": "$kategori", "jumlah": {"$sum": 1}}}
])

for data in hasil:
    print(data)


{'_id': 'Kopi', 'jumlah': 6}
{'_id': 'Makanan', 'jumlah': 6}


2.  Menampilkan Menu Berdasarkan Stok

In [20]:
# Menampilkan menu dengan stok di bawah 20
for data in db.menu.find(
    {"stok": {"$lt": 20}},
    {"_id": 0}
):
    print(data)


{'nama_menu': 'Cappuccino', 'harga': 25000, 'kategori': 'Kopi', 'tingkat_kepahitan': 3, 'stok': 15, 'status': 'Habis'}
{'nama_menu': 'Mocha', 'harga': 27000, 'kategori': 'Kopi', 'tingkat_kepahitan': 3, 'stok': 12, 'status': 'Tersedia'}
{'nama_menu': 'Mie Goreng', 'harga': 20000, 'kategori': 'Makanan', 'pedas_level': 3, 'stok': 15, 'status': 'Tersedia'}
{'nama_menu': 'Nasi Goreng', 'harga': 22000, 'kategori': 'Makanan', 'pedas_level': 2, 'stok': 18, 'status': 'Tersedia'}


3. Menampilkan Menu dengan Harga Tertinggi


In [22]:
# Menampilkan menu dengan harga tertinggi
hasil = db.menu.find_one(sort=[("harga", -1)])

print(hasil)


{'_id': ObjectId('6aeb83048b26acc03b13900d'), 'nama_menu': 'Mocha', 'harga': 27000, 'kategori': 'Kopi', 'tingkat_kepahitan': 3, 'stok': 12, 'status': 'Tersedia'}


In [23]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
